# LesioScan — 04 Validation Evaluation

**No training, no final test.** Loads the best ShuffleNet V2 fine-tuning checkpoint, computes validation classification metrics, confusion matrices, per-class recall, and prediction examples. Original images stay on Drive.

## 1. Google Drive dan source GitHub

In [ ]:
from pathlib import Path
import subprocess, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab: configure local paths below.')
REPO = Path('/content/lesioscan-ml')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Liex2810/lesioscan-ml.git',str(REPO)],check=True)
else:
    print('Repo found:',REPO)
print('Evaluation module present:',(REPO/'src/lesioscan/evaluation.py').is_file())
assert (REPO/'src/lesioscan/evaluation.py').is_file(), 'Tambahkan modul evaluation.py ke GitHub lalu clone/pull ulang.'

## 2. Konfigurasi lokasi dataset dan checkpoint

In [ ]:
ROOT=Path('/content/drive/MyDrive/LesioScan-AI')
DATASET=ROOT/'datasets/PAD-UFES-20'
SPLIT=ROOT/'experiments/splits/split.csv'
TRAIN_OUT=ROOT/'experiments/training/shufflenet_v2_baseline'
EVAL_OUT=TRAIN_OUT/'validation_evaluation'
for path in [DATASET,SPLIT,TRAIN_OUT/'fine_tuning/best_finetuning.pt']:
    print('✅' if path.exists() else '❌',path)
    assert path.exists(), f'Missing: {path}'
EVAL_OUT.mkdir(parents=True,exist_ok=True)
print('Report output:',EVAL_OUT)

## 3. Jalankan validation inference (tidak melakukan training)

`--trust-own-checkpoint` diperlukan hanya untuk konversi pertama checkpoint lama buatanmu yang mengandung NumPy RNG state. Jangan gunakan untuk file pihak lain.

In [ ]:
import subprocess,sys,time,threading,re
import ipywidgets as widgets
from IPython.display import display
bar=widgets.IntProgress(value=0,min=0,max=100,description='Evaluation:',layout=widgets.Layout(width='700px'))
status=widgets.HTML('Starting...')
display(bar,status)
command=[sys.executable,'-u','scripts/evaluate.py','--split',str(SPLIT),'--images',str(DATASET),
         '--training-out',str(TRAIN_OUT),'--out',str(EVAL_OUT),'--batch-size','16',
         '--trust-own-checkpoint']
process=subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                         text=True,bufsize=1,errors='replace')
log=[]
try:
    for line in process.stdout:
        line=line.strip()
        if line:
            log.append(line)
            if len(log)>60:log.pop(0)
            match=re.search(r'\[EVAL\s+(\d+)%\]\s*(.*)',line)
            if match:
                bar.value=int(match.group(1))
                status.value=match.group(2)
            elif 'Validation inference' in line:
                status.value='Validation inference running...'
    code=process.wait()
    if code:bar.bar_style='danger';print('\n'.join(log[-30:]));raise RuntimeError(f'Evaluation failed: {code}')
    bar.value=100;bar.bar_style='success';status.value='✅ Evaluation complete'
finally:
    if process.stdout:process.stdout.close()

## 4. Classification report dan ringkasan

In [ ]:
import pandas as pd,json
from IPython.display import display
TABLES=EVAL_OUT/'tables'; FIGURES=EVAL_OUT/'figures'
report=pd.read_csv(TABLES/'classification_report.csv',index_col=0)
display(report.round(4))
summary=json.loads((EVAL_OUT/'validation_summary.json').read_text())
print(json.dumps(summary,indent=2))

## 5. Confusion matrix dan recall

In [ ]:
from IPython.display import Image as DisplayImage,display
for name in ['confusion_matrix.png','confusion_matrix_normalized.png','recall_by_class.png']:
    print(name)
    display(DisplayImage(filename=str(FIGURES/name),width=800))
display(pd.read_csv(TABLES/'per_class_metrics.csv').round(4))

## 6. Contoh prediksi dan kesalahan

In [ ]:
display(DisplayImage(filename=str(FIGURES/'prediction_samples.png'),width=1100))
display(pd.read_csv(TABLES/'misclassification_pairs.csv').head(15))
display(pd.read_csv(TABLES/'high_confidence_errors.csv').head(10))
print('Predictions CSV:',TABLES/'validation_predictions.csv')

## 7. Audit output

In [ ]:
expected=[EVAL_OUT/'validation_summary.json',TABLES/'classification_report.csv',
          TABLES/'validation_predictions.csv',TABLES/'confusion_matrix.csv',
          FIGURES/'confusion_matrix.png',FIGURES/'recall_by_class.png',
          FIGURES/'prediction_samples.png']
for item in expected:
    print('✅' if item.is_file() else '❌',item)
assert all(x.is_file() for x in expected)
assert summary['test_set_evaluated'] is False
print('✅ Validation evaluation selesai; final test tetap tidak digunakan.')